# Standalone Mask Generation on Colab
Runs the cloth-agnostic mask pipeline independently.
Models are auto-downloaded from `zhengchong/CatVTON` on HuggingFace.

In [ ]:
# ── 1. Clone the repo ──────────────────────────────────────────────────────
!git clone https://github.com/usman9-ai/Virtual-Try-On.git
%cd Virtual-Try-On

In [ ]:
# ── 2. Install dependencies ────────────────────────────────────────────────
!pip install -r requirements.txt -q
!pip install fvcore av huggingface_hub -q

In [ ]:
# ── 3. Download models from HuggingFace (zhengchong/CatVTON) ──────────────
from huggingface_hub import snapshot_download
import os

repo_folder = snapshot_download(repo_id="zhengchong/CatVTON")
densepose_path = os.path.join(repo_folder, "DensePose")
schp_path      = os.path.join(repo_folder, "SCHP")

print(f"DensePose : {densepose_path}")
print(f"SCHP      : {schp_path}")

In [ ]:
# ── 4. Load the MaskGenerator ──────────────────────────────────────────────
import sys
sys.path.insert(0, "/content/Virtual-Try-On")  # make sure repo root is on path

from mask_generation import MaskGenerator, vis_mask

generator = MaskGenerator(
    densepose_ckpt=densepose_path,
    schp_ckpt=schp_path,
    device="cuda",
)
print("Models loaded.")

In [ ]:
# ── 5. Upload your image ───────────────────────────────────────────────────
from google.colab import files
from PIL import Image
import io

uploaded = files.upload()          # pick any person image from your machine
filename = list(uploaded.keys())[0]
person_image = Image.open(io.BytesIO(uploaded[filename])).convert("RGB")
person_image

In [ ]:
# ── 6. Generate mask ───────────────────────────────────────────────────────
# mask_type options: 'upper' | 'lower' | 'overall' | 'inner' | 'outer'
MASK_TYPE = "upper"

result = generator(person_image, mask_type=MASK_TYPE)

# Save outputs
os.makedirs("/content/mask_output", exist_ok=True)
result["mask"].save("/content/mask_output/mask.png")
result["densepose"].save("/content/mask_output/densepose.png")
result["schp_lip"].save("/content/mask_output/schp_lip.png")
result["schp_atr"].save("/content/mask_output/schp_atr.png")
vis_mask(person_image, result["mask"]).save("/content/mask_output/overlay.png")

print("Done. Outputs saved to /content/mask_output/")

In [ ]:
# ── 7. Visualise results ───────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 5, figsize=(20, 5))
titles = ["Original", "Mask", "Overlay", "DensePose", "SCHP (LIP)"]
images = [
    person_image,
    result["mask"],
    Image.open("/content/mask_output/overlay.png"),
    result["densepose"],
    result["schp_lip"],
]
for ax, img, title in zip(axes, images, titles):
    ax.imshow(img)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# ── 8. (Optional) Download all outputs as a zip ───────────────────────────
import shutil
shutil.make_archive("/content/mask_output", "zip", "/content/mask_output")
files.download("/content/mask_output.zip")